# Local advection-mismatch Vecchia conditional eigen diagnostic

This notebook runs the simulated smooth=0.5 advection-mismatch experiment. The default DGP uses the existing local simulated data with nugget=1. The fitted models all keep smoothness fixed at 0.5 and nugget fixed at 1, but force the advection parameters to true, zero, or large values.

In [ ]:
from pathlib import Path
import json
import shlex
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

REPO = Path('/Users/joonwonlee/Documents/GEMS_TCO-1')
DRIVER = REPO / 'Exercises/st_model/day/amarel_simulation/space_time/eigen_analysis/vecchia_conditional_eigen_sort_sim_smooth0p5_advection_mismatch_071126.py'
FULL_EIG_SCRIPT = REPO / 'Exercises/st_model/day/amarel_simulation/pure_space/eigen_diagnostic/eig_diag_real_july_st_full_eigen_matern_1of25_8h_070926.py'
RUN_STEM = 'sim_july_st_s05_vecchia_conditional_eigen_sort_advection_mismatch_071126'

PYTHON_EXE = Path('/opt/anaconda3/envs/faiss_env/bin/python')
if not PYTHON_EXE.exists():
    PYTHON_EXE = Path(sys.executable)

assert DRIVER.exists(), DRIVER
assert FULL_EIG_SCRIPT.exists(), FULL_EIG_SCRIPT
print('DRIVER:', DRIVER)
print('FULL_EIG_SCRIPT:', FULL_EIG_SCRIPT)
print('PYTHON_EXE:', PYTHON_EXE)

## Configure run

In [ ]:
DATA_ROOT = Path('/Users/joonwonlee/Documents/GEMS_DATA/simulation/july_st_circulant_realpattern_smooth0p5')
OUT_ROOT = REPO / 'outputs/summer_26/local_sim_s05_n1_vecchia_conditional_eigen_sort_advection_mismatch_071126'
FULL_EIG_OUT_ROOT = REPO / 'outputs/summer_26/local_sim_s05_n1_advection_mismatch_full_eigen_maxmin400_071126'

EXPERIMENT = 'advection_mismatch_dgp1'
YEAR = 2023
MONTH = 7
DAY = 1
DAYS = '0'
HOURS_PER_DAY = 8
TRUTH_NUGGET = 1.0
MAXMIN_POINTS_PER_HOUR = 400
FULL_EIG_MODEL_VARIANTS = 'advection_n1'

# Set True to rerun even when the combined summary already exists.
FORCE_RUN = False
FORCE_FULL_EIG = False

INPUT = DATA_ROOT / f'{YEAR}_july_st_circulant' / f'sim_july{YEAR}_st_circulant_gridded.pkl'
TRUTH_JSON = DATA_ROOT / f'{YEAR}_july_st_circulant' / f'sim_july{YEAR}_st_circulant_truth.json'
SUMMARY_CSV = OUT_ROOT / EXPERIMENT / f'{RUN_STEM}_{EXPERIMENT}_summary.csv'
PLOT_PNG = OUT_ROOT / EXPERIMENT / 'daily_plots' / f'year_{YEAR}' / f'sim_{YEAR}_day01_vecchia_conditional_eigen_sort_comparison.png'
FULL_EIG_DIR = FULL_EIG_OUT_ROOT / f'year{YEAR}_day{DAY:02d}_maxmin{MAXMIN_POINTS_PER_HOUR}_per_hour'
FULL_EIG_SUMMARY_CSV = FULL_EIG_DIR / 'st_full_eigen_summary.csv'
FULL_EIG_CURVES_CSV = FULL_EIG_DIR / 'st_full_eigen_curves.csv'
FULL_EIG_PLOT_PNG = FULL_EIG_DIR / 'st_full_eigen_model_comparison.png'

print('DATA_ROOT:', DATA_ROOT)
print('OUT_ROOT:', OUT_ROOT)
print('FULL_EIG_OUT_ROOT:', FULL_EIG_OUT_ROOT)
print('INPUT:', INPUT)
print('TRUTH_JSON:', TRUTH_JSON)
print('SUMMARY_CSV:', SUMMARY_CSV)
print('PLOT_PNG:', PLOT_PNG)
print('FULL_EIG_SUMMARY_CSV:', FULL_EIG_SUMMARY_CSV)
print('FULL_EIG_PLOT_PNG:', FULL_EIG_PLOT_PNG)

## Validate simulated data

In [ ]:
if not INPUT.exists():
    raise FileNotFoundError(f'Missing simulated pickle: {INPUT}')
if not TRUTH_JSON.exists():
    raise FileNotFoundError(f'Missing truth JSON: {TRUTH_JSON}')

truth = json.loads(TRUTH_JSON.read_text())
truth_view = {k: truth.get(k) for k in ['smooth', 'nugget', 'sigmasq', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'n_hours']}
print(json.dumps(truth_view, indent=2))

if abs(float(truth['smooth']) - 0.5) > 1e-12:
    raise RuntimeError(f'Expected smooth=0.5, got {truth["smooth"]}')
if abs(float(truth['nugget']) - TRUTH_NUGGET) > 1e-12:
    raise RuntimeError(f'Expected nugget={TRUTH_NUGGET}, got {truth["nugget"]}')

obj = pd.read_pickle(INPUT)
print('hour keys:', len(obj))
print('first 8 keys:', list(obj.keys())[:8])
first_df = next(iter(obj.values()))
print('first hour shape:', first_df.shape)
display(first_df.head())

## Run advection-mismatch experiment

The three fitted variants are true advection, zero advection, and large fixed advection `(0.5, 0.5)`. Only `sigmasq` and the range parameters are optimized; the advection parameters are fixed in the driver.

In [ ]:
cmd = [
    str(PYTHON_EXE),
    str(DRIVER),
    '--experiment', EXPERIMENT,
    '--isolate-models',
    '--split-fit-diagnostic',
    '--data-root', str(DATA_ROOT),
    '--out-root', str(OUT_ROOT),
    '--years', str(YEAR),
    '--month', str(MONTH),
    '--days', DAYS,
    '--hours-per-day', str(HOURS_PER_DAY),
    '--truth-nugget', str(TRUTH_NUGGET),
    '--sim-kind', 'gridded',
    '--lat-range=-3,2',
    '--lon-range=121,131',
    '--keep-exact-loc',
    '--real-reference-advec-lon-abs', '0.126',
    '--daily-stride', '2',
    '--target-chunk-size', '16',
    '--diag-chunk-size', '64',
    '--min-target-points', '1',
    '--spline-n-points', '4000',
    '--spline-r-max', '30.0',
    '--lbfgs-lr', '1.0',
    '--lbfgs-steps', '5',
    '--lbfgs-eval', '20',
    '--lbfgs-history', '10',
    '--grad-tol', '1e-5',
    '--device', 'cpu',
    '--cuda-fallback', 'cpu',
    '--resample-grid', '200',
    '--save-daily-curves',
    '--suppress-fit-prints',
]

if SUMMARY_CSV.exists() and PLOT_PNG.exists() and not FORCE_RUN:
    print('Reusing existing outputs. Set FORCE_RUN=True to rerun.')
    print(SUMMARY_CSV)
else:
    print(shlex.join(cmd))
    proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f'Advection mismatch experiment failed with exit code {ret}')

## Show summary and plot

In [ ]:
if not SUMMARY_CSV.exists():
    raise FileNotFoundError(SUMMARY_CSV)
summary = pd.read_csv(SUMMARY_CSV)
cols = [
    'status', 'day', 'model_variant', 'model_label', 'vecchia_loss_per_obs',
    'fixed_advec_lat', 'fixed_advec_lon', 'est_advec_lat', 'est_advec_lon',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time', 'est_nugget',
    'mean_y2', 'max_abs_bridge_scaled'
]
cols = [c for c in cols if c in summary.columns]
display(summary[cols])

if PLOT_PNG.exists():
    display(Image(filename=str(PLOT_PNG)))
else:
    print('Plot not found:', PLOT_PNG)

## Full eigen diagnostic on first 400 max-min spatial points per hour

This section uses the full-data Vecchia fit summary above, then runs a dense full ST eigendecomposition on the first 400 max-min spatial locations from the first hour, repeated across all 8 hours. Total size is about `400 * 8 = 3200` observations. Results are saved under `FULL_EIG_OUT_ROOT` and loaded if they already exist.

In [ ]:
if not SUMMARY_CSV.exists():
    raise FileNotFoundError(f'Run/load the Vecchia summary first: {SUMMARY_CSV}')

full_eig_cmd = [
    str(PYTHON_EXE),
    str(FULL_EIG_SCRIPT),
    '--input', str(INPUT),
    '--vecchia-summary', str(SUMMARY_CSV),
    '--output-root', str(FULL_EIG_OUT_ROOT),
    '--year', str(YEAR),
    '--month', str(MONTH),
    '--day', str(DAY),
    '--hours-per-day', str(HOURS_PER_DAY),
    '--subset-mode', 'maxmin',
    '--maxmin-points-per-hour', str(MAXMIN_POINTS_PER_HOUR),
    '--model-variants', FULL_EIG_MODEL_VARIANTS,
    '--save-selected-points',
]

if FULL_EIG_SUMMARY_CSV.exists() and FULL_EIG_PLOT_PNG.exists() and not FORCE_FULL_EIG:
    print('Reusing existing full-eigen maxmin outputs. Set FORCE_FULL_EIG=True to rerun.')
    print(FULL_EIG_SUMMARY_CSV)
else:
    print(shlex.join(full_eig_cmd))
    proc = subprocess.Popen(full_eig_cmd, cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f'Full eigen maxmin diagnostic failed with exit code {ret}')

## Show maxmin full-eigen results

In [ ]:
if not FULL_EIG_SUMMARY_CSV.exists():
    raise FileNotFoundError(FULL_EIG_SUMMARY_CSV)
full_eig_summary = pd.read_csv(FULL_EIG_SUMMARY_CSV)
full_cols = [
    'variant', 'label', 'source_model_variant', 'source_vecchia_loss_per_obs',
    'loss_per_obs', 'score2_per_df', 'max_abs_bridge', 'n_obs', 'residual_df',
    'subset_mode', 'maxmin_points_per_hour',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time',
    'est_advec_lat', 'est_advec_lon', 'est_nugget'
]
full_cols = [c for c in full_cols if c in full_eig_summary.columns]
display(full_eig_summary[full_cols])

print('FULL_EIG_SUMMARY_CSV:', FULL_EIG_SUMMARY_CSV)
print('FULL_EIG_CURVES_CSV:', FULL_EIG_CURVES_CSV)
print('FULL_EIG_PLOT_PNG:', FULL_EIG_PLOT_PNG)
if FULL_EIG_PLOT_PNG.exists():
    display(Image(filename=str(FULL_EIG_PLOT_PNG)))
else:
    print('Plot not found:', FULL_EIG_PLOT_PNG)

## Notes

For the nugget=0 DGP version, change `EXPERIMENT` to `advection_mismatch`, `TRUTH_NUGGET` to `0.0`, and point `DATA_ROOT` to a validated nugget0 simulated data root.